<p align="center">
  <a href="https://colab.research.google.com/github/samuel-c-santos/opensr-sentinel-pipeline/blob/main/OpenSR_Producao_Poligono_Colab.ipynb" target="_blank">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir no Google Colab" width="180">
  </a>
</p>

# ESA OpenSR - Super-Resolucao de Sentinel-2 por Poligono (AOI)
## Processamento em Lote com Janela Deslizante (10m para 2.5m / 4x)

Este notebook executa o fluxo operacional de super-resolucao espacial do sensor Sentinel-2 sobre Areas de Interesse (AOI) delimitadas por Shapefile, baseado na mesma rotina de ingestao geoespacial do NGEO Satellite Downloader com exportacao direta para o Google Drive.

### Caracteristicas do Fluxo:
- **Ingestao Vetorial via Shapefile**: Interface de upload de arquivo compactado (.zip) com conversao direta para objeto de geometria do Earth Engine (geemap e geopandas).
- **Mascara e Composicao Inteligente**: Filtro de nuvens sobre a regiao de interesse (ROI) e extracao das 4 bandas de 10 metros: B04 (Vermelho), B03 (Verde), B02 (Azul) e B08 (NIR).
- **Processamento por Janelas Deslizantes (Tiling)**: Subdivisao automatica em blocos com sobreposicao (overlap) e ponderacao por funcao Hanning, prevenindo esgotamento de memoria VRAM e eliminando descontinuidades nas emendas.
- **Exportacao Georreferenciada**: O produto final e gerado em 2.5 metros por pixel com matriz de transformacao espacial corrigida e gravado no Google Drive.


## 1. Instalacao das Dependencias

Execute esta celula uma vez por sessao do Colab. Certifique-se de que a aceleracao por hardware esta configurada para GPU (T4).

In [ ]:
# Instalacao dos pacotes necessarios
!pip install -q opensr-model "opensr-utils[model]" omegaconf rasterio geopandas geemap earthengine-api

## 2. Autenticacao e Inicializacao (Earth Engine e Google Drive)
Inicializa o projeto `sentinel-landsat-download` e monta o Google Drive para gravacao dos resultados.

In [ ]:
import ee
import os
import shutil
from google.colab import drive

# 1. Autenticacao e Inicializacao Earth Engine
GEE_PROJECT = "sentinel-landsat-download"
try:
    ee.Initialize(project=GEE_PROJECT)
    print(f"Google Earth Engine inicializado no projeto: {GEE_PROJECT}")
except Exception as e:
    ee.Authenticate()
    ee.Initialize(project=GEE_PROJECT)
    print(f"Google Earth Engine autenticado e inicializado: {GEE_PROJECT}")

# 2. Montagem do Google Drive
drive.mount("/content/drive")
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/Satellite_Downloads/OpenSR_2_5m"
os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
print(f"Pasta de destino no Google Drive: {DRIVE_OUTPUT_DIR}")

# 3. Diretorios locais da sessao
WORK_DIR = "/content/data"
SHAPEFILE_DIR = "/content/shapefile"
os.makedirs(WORK_DIR, exist_ok=True)
os.makedirs(SHAPEFILE_DIR, exist_ok=True)

# Variavel global da regiao do Shapefile
SHAPEFILE_REGION = None

## 3. Upload do Shapefile e Deteccao da Area de Interesse (AOI)
Clique no botao abaixo e selecione o arquivo `.zip` contendo os arquivos do seu Shapefile (`.shp`, `.shx`, `.dbf`, `.prj`). O sistema extrai e converte a geometria para o Earth Engine automaticamente.

In [ ]:
import zipfile
import json as _json
import math
import geemap
import geopandas as gpd
from google.colab import files
import ipywidgets as widgets
from IPython.display import display, clear_output

upload_output = widgets.Output()

def estimate_pixel_count(region, scale=10):
    bounds = region.bounds().coordinates().get(0).getInfo()
    xs = [pt[0] for pt in bounds]
    ys = [pt[1] for pt in bounds]
    lat_mid = (min(ys) + max(ys)) / 2
    m_per_deg_lon = 111320 * math.cos(math.radians(lat_mid))
    m_per_deg_lat = 110540
    width_m = (max(xs) - min(xs)) * m_per_deg_lon
    height_m = (max(ys) - min(ys)) * m_per_deg_lat
    return int((width_m / scale) * (height_m / scale))

def on_upload_shapefile_clicked(b):
    global SHAPEFILE_REGION
    with upload_output:
        clear_output()
        print("Selecione o arquivo .zip com o Shapefile da sua Area de Interesse...")
        if os.path.exists(SHAPEFILE_DIR):
            shutil.rmtree(SHAPEFILE_DIR)
        os.makedirs(SHAPEFILE_DIR, exist_ok=True)

        uploaded = files.upload()
        zip_filename = None
        for fname in uploaded.keys():
            if fname.lower().endswith(".zip"):
                zip_filename = fname
                break

        if not zip_filename:
            print("Nenhum arquivo .zip identificado no upload.")
            return

        with zipfile.ZipFile(zip_filename, "r") as zip_ref:
            zip_ref.extractall(SHAPEFILE_DIR)
        print(f"Arquivo {zip_filename} extraido com sucesso!")

        shp_candidates = [f for f in os.listdir(SHAPEFILE_DIR) if f.lower().endswith(".shp")]
        if not shp_candidates:
            print("Nenhum arquivo com extensao .shp encontrado dentro do .zip.")
            return

        shp_path = os.path.join(SHAPEFILE_DIR, shp_candidates[0])
        print(f"Carregando vetor: {shp_candidates[0]}...")

        feature = None
        try:
            feature = geemap.shp_to_ee(shp_path)
        except Exception as ee_err:
            print(f"Aviso na conversao direta com geemap: {ee_err}. Utilizando geopandas...")

        if feature is None:
            try:
                gdf = gpd.read_file(shp_path)
                if gdf.crs is None:
                    gdf = gdf.set_crs("EPSG:4326")
                elif gdf.crs.to_string() != "EPSG:4326":
                    gdf = gdf.to_crs("EPSG:4326")
                feature = ee.FeatureCollection(_json.loads(gdf.to_json()))
            except Exception as gpd_err:
                print(f"Erro ao processar arquivo com geopandas: {gpd_err}")
                return

        SHAPEFILE_REGION = feature.geometry()
        bounds = SHAPEFILE_REGION.bounds().coordinates().get(0).getInfo()
        xs = [pt[0] for pt in bounds]
        ys = [pt[1] for pt in bounds]

        print(f"Limites geograficos da AOI:")
        print(f"  Longitude: [{min(xs):.5f}, {max(xs):.5f}]")
        print(f"  Latitude:  [{min(ys):.5f}, {max(ys):.5f}]")

        px_10m = estimate_pixel_count(SHAPEFILE_REGION, scale=10)
        px_2_5m = px_10m * 16
        print(f"Estimativa de dimensoes da regiao:")
        print(f"  Resolucao nativa (10m): {px_10m:,} pixels")
        print(f"  Resolucao alvo (2.5m):  {px_2_5m:,} pixels")
        print("Geometria pronta para pesquisa de imagens.")

upload_btn = widgets.Button(
    description="Upload Shapefile (.zip)",
    button_style="info",
    layout=widgets.Layout(width="320px", height="42px")
)
upload_btn.on_click(on_upload_shapefile_clicked)
display(upload_btn, upload_output)

## 4. Selecao da Cena e Aquisicao das Bandas Sentinel-2 (10m)
Busca a imagem com menor cobertura de nuvens sobre a geometria carregada do Shapefile e faz o recorte das 4 bandas VNIR de 10 metros: **B4 (Red), B3 (Green), B2 (Blue) e B8 (NIR)**.

In [ ]:
import urllib.request
import rasterio

if SHAPEFILE_REGION is None:
    raise ValueError("Nenhum Shapefile foi carregado. Execute a celula 3 e faca o upload do arquivo .zip.")

# Parametros de data e filtro de nuvens
DATA_INICIAL = "2025-08-01"
DATA_FINAL = "2026-03-01"
LIMITE_NUVENS = 30

aoi_bounds = SHAPEFILE_REGION.bounds()

collection = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(SHAPEFILE_REGION)
    .filterDate(DATA_INICIAL, DATA_FINAL)
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", LIMITE_NUVENS))
    .sort("CLOUDY_PIXEL_PERCENTAGE")
)

n_imgs = collection.size().getInfo()
if n_imgs == 0:
    print("Nenhuma cena com < 30% de nuvens no periodo especificado. Buscando menor cobertura anual...")
    collection = (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(SHAPEFILE_REGION)
        .sort("CLOUDY_PIXEL_PERCENTAGE")
    )
    s2_image = ee.Image(collection.first())
else:
    s2_image = ee.Image(collection.first())
    print(f"Encontradas {n_imgs} imagens limpas no acervo. Selecionando a mais adequada.")

scene_date = s2_image.date().format("YYYY-MM-dd").getInfo()
scene_id = s2_image.get("system:index").getInfo()
scene_cloud = s2_image.get("CLOUDY_PIXEL_PERCENTAGE").getInfo()

print(f"Cena identificada: {scene_id}")
print(f"Data de aquisicao: {scene_date}")
print(f"Percentual de nuvens: {scene_cloud:.2f}%")

# Recorte nas bandas VNIR de 10m
s2_bands = s2_image.select(["B4", "B3", "B2", "B8"]).clip(aoi_bounds)

input_tif_path = os.path.join(WORK_DIR, f"input_s2_10m_{scene_date}.tif")
download_url = s2_bands.getDownloadURL({
    "scale": 10,
    "crs": "EPSG:3857",
    "region": aoi_bounds,
    "format": "GEO_TIFF"
})

print("Baixando imagem Sentinel-2 recortada para o polígono...")
urllib.request.urlretrieve(download_url, input_tif_path)

with rasterio.open(input_tif_path) as src:
    lr_shape = src.shape
    lr_count = src.count
    lr_profile = src.profile
    print(f"Download do raster concluido com sucesso!")
    print(f"Matriz LR (10m): {lr_count} bandas, {lr_shape[0]} linhas x {lr_shape[1]} colunas")

## 5. Carregamento do Modelo ESA OpenSR (LDSR-S2)
Inicializa o modelo de Difusao Latente treinado pela ESA para as bandas VNIR de Sentinel-2.

In [ ]:
from io import StringIO
import requests
import torch
from omegaconf import OmegaConf
import opensr_model

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Dispositivo de processamento: {device.upper()}")
if device == "cpu":
    print("AVISO: GPU nao identificada. Recomenda-se ativar GPU T4 para processamento de poligonos.")

config_url = "https://raw.githubusercontent.com/ESAOpenSR/opensr-model/refs/heads/main/opensr_model/configs/config_10m.yaml"
response = requests.get(config_url)
config = OmegaConf.load(StringIO(response.text))

print("Carregando pesos pre-treinados do LDSR-S2 da ESA no Hugging Face...")
model = opensr_model.SRLatentDiffusion(config, device=device)
model.load_pretrained(config.ckpt_version)
model.eval()
print("Modelo ESA OpenSR pronto para inferencia.")

## 6. Super-Resolucao por Janelas Deslizantes (Tiling com Overlap)
Subdivide a extensao em blocos regulares de 128x128 pixels de baixa resolucao com sobreposicao de 16 pixels. A reconstrucao aplica uma funcao de ponderacao Hanning 2D nas bordas, eliminando linhas de costura visiveis e mantendo o consumo de VRAM estavel.

In [ ]:
import numpy as np
import math

def process_large_raster_tiled(input_path, model, device, window_size=128, overlap=16, scale=4):
    with rasterio.open(input_path) as src:
        img = src.read()  # (C, H, W)
        profile = src.profile

    c, orig_h, orig_w = img.shape

    # Garante dimensoes minimas de window_size (128x128) via padding reflexivo
    # prevenindo incompatibilidades de shape na UNet caso o poligono seja pequeno
    pad_h = max(0, window_size - orig_h)
    pad_w = max(0, window_size - orig_w)
    if pad_h > 0 or pad_w > 0:
        img = np.pad(img, ((0, 0), (0, pad_h), (0, pad_w)), mode="reflect")
        c, h, w = img.shape
    else:
        h, w = orig_h, orig_w

    sr_h, sr_w = h * scale, w * scale

    output = np.zeros((c, sr_h, sr_w), dtype=np.float32)
    weight = np.zeros((sr_h, sr_w), dtype=np.float32)

    step = window_size - overlap
    y_steps = max(1, math.ceil((h - window_size) / step) + 1) if h > window_size else 1
    x_steps = max(1, math.ceil((w - window_size) / step) + 1) if w > window_size else 1
    total_tiles = y_steps * x_steps

    print(f"Iniciando processamento: grade de {y_steps} x {x_steps} blocos (total: {total_tiles})...")

    # Janela Hanning 2D para fusao suave
    win_sr = window_size * scale
    han_1d = np.hanning(win_sr)
    win_weight = np.outer(han_1d, han_1d).astype(np.float32)
    win_weight = np.clip(win_weight, 0.05, 1.0)

    count = 0
    for yi in range(y_steps):
        y1 = min(yi * step, h - window_size)
        y2 = y1 + window_size
        for xi in range(x_steps):
            x1 = min(xi * step, w - window_size)
            x2 = x1 + window_size

            patch = img[:, y1:y2, x1:x2].astype(np.float32)
            patch_tensor = torch.from_numpy(patch).float().unsqueeze(0) / 10000.0
            patch_tensor = patch_tensor.to(device)

            with torch.no_grad():
                sr_patch = model(patch_tensor)

            sr_patch_np = sr_patch.squeeze(0).cpu().numpy()

            sy1, sy2 = y1 * scale, y2 * scale
            sx1, sx2 = x1 * scale, x2 * scale
            cur_h, cur_w = sy2 - sy1, sx2 - sx1

            cur_weight = win_weight[:cur_h, :cur_w]
            for band in range(c):
                output[band, sy1:sy2, sx1:sx2] += sr_patch_np[band, :cur_h, :cur_w] * cur_weight
            weight[sy1:sy2, sx1:sx2] += cur_weight

            count += 1
            if count % 5 == 0 or count == total_tiles:
                print(f"  Progresso: {count}/{total_tiles} blocos processados.")

    weight = np.maximum(weight, 1e-6)
    for band in range(c):
        output[band] /= weight

    output = np.clip(output * 10000.0, 0, 10000).astype(np.uint16)
    output = output[:, :orig_h * scale, :orig_w * scale]
    return output, profile

sr_matrix, in_profile = process_large_raster_tiled(input_tif_path, model, device)
print("Processamento por janelas deslizantes finalizado com sucesso!")
print(f"Matriz SR (2.5m): {sr_matrix.shape[0]} bandas, {sr_matrix.shape[1]} linhas x {sr_matrix.shape[2]} colunas")

## 7. Gravacao do GeoTIFF de 2.5m e Exportacao para o Google Drive
Ajusta a matriz afim (`Affine transform`) para a resolucao de 2.5 metros por pixel (fator 4x) e salva o produto final diretamente na pasta do Google Drive.

In [ ]:
from rasterio.transform import Affine

scale_factor = 4
orig_transform = in_profile["transform"]
new_transform = Affine(
    orig_transform.a / scale_factor, orig_transform.b, orig_transform.c,
    orig_transform.d, orig_transform.e / scale_factor, orig_transform.f
)

out_profile = in_profile.copy()
out_profile.update({
    "height": sr_matrix.shape[1],
    "width": sr_matrix.shape[2],
    "transform": new_transform,
    "dtype": "uint16",
    "nodata": 0
})

shp_name_base = os.path.splitext(os.path.basename(input_tif_path))[0]
out_filename = f"OpenSR_2_5m_{scene_date}_{shp_name_base}.tif"
local_tif_output = os.path.join(WORK_DIR, out_filename)
drive_tif_output = os.path.join(DRIVE_OUTPUT_DIR, out_filename)

# 1. Grava no disco da sessao
with rasterio.open(local_tif_output, "w", **out_profile) as dst:
    dst.write(sr_matrix)
print(f"GeoTIFF local gravado: {local_tif_output}")

# 2. Copia para o Google Drive
shutil.copy2(local_tif_output, drive_tif_output)
print(f"Arquivo exportado para o Google Drive:")
print(f"  Caminho: {drive_tif_output}")

## 8. Inspecao Visual Comparativa da AOI (10m vs 2.5m)

In [ ]:
import matplotlib.pyplot as plt

def to_rgb(img_bands, r_idx, g_idx, b_idx, p_min=2, p_max=98):
    rgb = np.stack([img_bands[r_idx], img_bands[g_idx], img_bands[b_idx]], axis=-1).astype(float)
    for c in range(3):
        band = rgb[..., c]
        v_min, v_max = np.percentile(band, (p_min, p_max))
        if v_max > v_min:
            rgb[..., c] = np.clip((band - v_min) / (v_max - v_min), 0, 1)
    return rgb

with rasterio.open(input_tif_path) as src:
    orig_lr = src.read()

rgb_lr = to_rgb(orig_lr, 0, 1, 2)  # B4, B3, B2
rgb_sr = to_rgb(sr_matrix, 0, 1, 2)

nir_lr = to_rgb(orig_lr, 3, 0, 1)  # B8, B4, B3 (Falsa-cor)
nir_sr = to_rgb(sr_matrix, 3, 0, 1)

fig, axes = plt.subplots(2, 2, figsize=(14, 14))

axes[0, 0].imshow(rgb_lr)
axes[0, 0].set_title(f"Sentinel-2 Entrada (10m/px) - RGB\nData: {scene_date}", fontsize=11, fontweight="bold")
axes[0, 0].axis("off")

axes[0, 1].imshow(rgb_sr)
axes[0, 1].set_title("ESA OpenSR (2.5m/px) - RGB\nReconstrucao Tiled 4x", fontsize=11, fontweight="bold", color="darkgreen")
axes[0, 1].axis("off")

axes[1, 0].imshow(nir_lr)
axes[1, 0].set_title("Sentinel-2 Entrada (10m/px) - Falsa Cor (NIR)", fontsize=11, fontweight="bold")
axes[1, 0].axis("off")

axes[1, 1].imshow(nir_sr)
axes[1, 1].set_title("ESA OpenSR (2.5m/px) - Falsa Cor (NIR)", fontsize=11, fontweight="bold", color="darkgreen")
axes[1, 1].axis("off")

plt.tight_layout()
plt.show()